## 线性回归的简洁实现

In [1]:
import numpy as np
import torch
from torch.utils import data

def synthetic_data(w, b, num_examples):  
    """生成y=Xw+b+噪声"""
    X = torch.normal(0, 1, (num_examples, len(w)))
    y = torch.matmul(X, w) + b
    y += torch.normal(0, 0.01, y.shape)
    return X, y.reshape((-1, 1))

true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = synthetic_data(true_w, true_b, 1000)

In [2]:
def load_array(data_arrays, batch_size, is_train=True):  
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

batch_size = 10
data_iter = load_array((features, labels), batch_size)

In [3]:
next(iter(data_iter))

[tensor([[-0.2097, -1.5151],
         [-0.6756,  0.6741],
         [ 0.2451,  0.7706],
         [-0.3680,  0.3523],
         [ 0.2153,  0.2771],
         [-0.1971, -0.6741],
         [-0.3945, -0.1562],
         [ 1.0460, -0.5706],
         [-0.7813, -0.2748],
         [ 0.4349, -0.4139]]),
 tensor([[8.9353],
         [0.5520],
         [2.0424],
         [2.2691],
         [3.6954],
         [6.0936],
         [3.9539],
         [8.2333],
         [3.5693],
         [6.4719]])]

In [4]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

In [5]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

In [6]:
loss = nn.MSELoss()

In [7]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

In [8]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000329
epoch 2, loss 0.000095
epoch 3, loss 0.000094


In [9]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([-0.0002,  0.0003])
b的估计误差： tensor([9.5844e-05])
